# Aither — Phi-2 QLoRA Fine-Tuning
**Mental Health | Psychology | Medicine**

Fine-tunes Microsoft Phi-2 (2.7B) using QLoRA (4-bit NF4 + LoRA rank 32) on 7 HuggingFace datasets + Aither's therapeutic module knowledge.

All 4 Aither runtime modules baked into training:
- `safety.py` — Crisis detection across 5 severity levels
- `rag.py` — 40+ therapeutic documents (CBT, DBT, mindfulness, anxiety, depression, crisis, relationships, self-esteem)
- `emotion.py` — 4 response tones (assertive, tender, empathetic, neutral)
- `memory.py` — Conversation summarization for context compaction

In [ ]:
# Run this cell ONCE, then skip it after runtime restarts
!pip install -q -U transformers datasets accelerate peft bitsandbytes trl scipy einops numpy
import os; os.kill(os.getpid(), 9)

In [ ]:
import torch, os
assert torch.cuda.is_available(), "No GPU — go to Runtime > Change runtime type > GPU"
print(f"PyTorch {torch.__version__} | GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.mem_get_info(0)[1]/1e9:.1f} GB")

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Config:
    model: str = "microsoft/phi-2"
    context_window: int = 2048

    # Training
    batch_size: int = 2
    learning_rate: float = 5e-5
    epochs: int = 2
    gradient_acc_steps: int = 8       # effective batch = 2 * 8 = 16
    weight_decay: float = 0.01
    warmup_steps: int = 100
    max_samples_per_dataset: int = 10000
    max_total_samples: int = 50000

    # QLoRA
    lora_rank: int = 32
    lora_alpha: int = 64
    lora_dropout: float = 0.05
    target_modules: list = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "dense", "fc1", "fc2"
    ])

    # Eval & saving
    eval_split: float = 0.05
    eval_steps: int = 25
    logging_steps: int = 25
    save_steps: int = 25

    # Paths
    output_dir: str = "./aither_trained"
    merged_dir: str = "./aither_merged"

config = Config()
print(f"Model: {config.model} | LR: {config.learning_rate} | Epochs: {config.epochs} | LoRA r={config.lora_rank} a={config.lora_alpha}")
print(f"Effective batch: {config.batch_size * config.gradient_acc_steps} | Targets: {config.target_modules}")

## Dataset Loading

In [ ]:
from datasets import load_dataset, Dataset
import random

DATASETS = {
    "Amod/mental_health_counseling_conversations":       {"type": "conversations", "domain": "mental_health"},
    "heliosbrahma/mental_health_chatbot_dataset":        {"type": "text",          "domain": "mental_health"},
    "mpingale/mental-health-chat-dataset":               {"type": "qa",            "domain": "mental_health"},
    "nbertagnolli/counsel-chat":                         {"type": "qa",            "domain": "mental_health"},
    "ruslanmv/ai-medical-chatbot":                       {"type": "medical",       "domain": "medicine"},
    "lavita/ChatDoctor-HealthCareMagic-100k":            {"type": "instruction",   "domain": "medicine"},
    "alexandreteles/mental-health-conversational-data":  {"type": "conversations", "domain": "psychology"},
}

def fmt_conversations(ex):
    c = ex["conversations"]
    r = ""
    for t in c:
        role = t["role"].lower()
        if role in ("human", "user", "patient"):         r += "<|user|>" + t["content"].strip()
        elif role in ("assistant", "gpt", "therapist", "counselor", "doctor"): r += "<|assistant|>" + t["content"].strip()
    return r if "<|user|>" in r and "<|assistant|>" in r else None

def fmt_text(ex):
    t = ex["text"].replace("<HUMAN>:", "<|user|>").replace("<ASSISTANT>:", "<|assistant|>")
    return t if "<|user|>" in t and "<|assistant|>" in t else None

def fmt_qa(ex):
    q = ((ex.get("questionTitle") or "") + " " + (ex.get("questionText") or "")).strip()
    a = (ex.get("answerText") or "").strip()
    return f"<|user|>{q}<|assistant|>{a}" if a else None

def fmt_medical(ex):
    ctx = (ex.get("Description") or "").strip()
    p = (ex.get("Patient") or "").strip()
    d = (ex.get("Doctor") or "").strip()
    prompt = f"{ctx} {p}".strip() if ctx else p
    return f"<|user|>{prompt}<|assistant|>{d}" if d else None

def fmt_instruction(ex):
    inst = (ex.get("instruction") or "").strip()
    inp = (ex.get("input") or "").strip()
    out = (ex.get("output") or "").strip()
    prompt = f"{inst} {inp}".strip() if inp else inst
    return f"<|user|>{prompt}<|assistant|>{out}" if out else None

FMT = {"conversations": fmt_conversations, "text": fmt_text, "qa": fmt_qa, "medical": fmt_medical, "instruction": fmt_instruction}

In [ ]:
all_texts = []
domain_counts = {"mental_health": 0, "medicine": 0, "psychology": 0}

for name, info in DATASETS.items():
    fmt = FMT[info["type"]]
    domain = info["domain"]
    print(f"Loading [{domain}] {name}...")
    try:
        data = load_dataset(name, split="train")
    except Exception as e:
        print(f"  SKIP: {e}"); continue

    count = 0
    for ex in data:
        if count >= config.max_samples_per_dataset or len(all_texts) >= config.max_total_samples: break
        try:
            text = fmt(ex)
        except (KeyError, TypeError, AttributeError):
            continue
        if text and len(text) > 50:
            all_texts.append({"text": text})
            count += 1
            domain_counts[domain] += 1
    print(f"  {count} samples (total: {len(all_texts)})")

print(f"\nDataset totals: {len(all_texts)} | MH: {domain_counts['mental_health']} | Med: {domain_counts['medicine']} | Psych: {domain_counts['psychology']}")

## Aither Module Knowledge Injection
Synthetic training examples from all 4 runtime modules — repeated 5x and shuffled into the dataset.

In [ ]:
module_examples = []

RAG_KNOWLEDGE = {
    "Cognitive Behavioral Therapy": [
        "Cognitive restructuring helps identify and challenge negative thought patterns by examining evidence for and against automatic thoughts.",
        "Behavioral activation encourages engaging in meaningful activities to combat withdrawal and low mood, starting with small achievable goals.",
        "Thought records track situations, emotions, automatic thoughts, and alternative perspectives to build awareness of thinking patterns.",
        "Socratic questioning guides the user to examine their beliefs by asking what evidence supports or contradicts their thoughts.",
        "Cognitive distortions include all-or-nothing thinking, catastrophizing, mind reading, and emotional reasoning which can be identified and reframed.",
    ],
    "Dialectical Behavior Therapy": [
        "Distress tolerance skills like TIPP (Temperature, Intense exercise, Paced breathing, Progressive relaxation) help manage acute emotional crises.",
        "Emotion regulation involves identifying and labeling emotions, understanding their function, and reducing vulnerability to negative emotions.",
        "Interpersonal effectiveness skills help maintain self-respect and relationships while making requests or saying no using the DEAR MAN technique.",
        "Radical acceptance means fully accepting reality as it is without judgment, reducing suffering caused by fighting unchangeable circumstances.",
        "The wise mind concept balances emotional mind and rational mind to make decisions that honor both feelings and logic.",
    ],
    "Mindfulness and Grounding": [
        "Box breathing involves inhaling for 4 counts, holding for 4 counts, exhaling for 4 counts, and holding for 4 counts to activate the parasympathetic nervous system.",
        "The 5-4-3-2-1 grounding technique uses the senses: name 5 things you see, 4 you hear, 3 you touch, 2 you smell, and 1 you taste.",
        "Body scan meditation brings awareness to each part of the body from toes to head, noticing sensations without judgment to release tension.",
        "Mindful observation involves focusing attention on a single object or sensation for several minutes, gently returning focus when the mind wanders.",
        "Progressive muscle relaxation systematically tenses and releases muscle groups to reduce physical tension associated with stress and anxiety.",
    ],
    "Anxiety Management": [
        "Exposure therapy gradually confronts feared situations in a controlled way, starting with less anxiety-provoking scenarios and building tolerance.",
        "Worry time is a scheduled period to process anxious thoughts, training the mind to postpone worry and reduce its intrusion throughout the day.",
        "Anxiety psychoeducation explains that anxiety is a normal protective response that becomes problematic when activated disproportionately to actual threat.",
        "Safety behaviors and avoidance maintain anxiety by preventing the person from learning that feared outcomes are unlikely or manageable.",
        "Challenging catastrophic thinking involves asking what is the worst case, best case, and most likely outcome to gain realistic perspective.",
    ],
    "Depression Support": [
        "Activity scheduling combats depression by planning pleasurable and mastery activities throughout the week to rebuild engagement and accomplishment.",
        "Behavioral experiments test negative predictions by trying new behaviors and observing actual outcomes versus expected outcomes.",
        "Sleep hygiene practices include maintaining consistent sleep and wake times, limiting screen time before bed, and creating a restful environment.",
        "Social connection even in small doses counteracts isolation, starting with brief low-pressure interactions and gradually increasing social engagement.",
        "Self-compassion practices involve treating yourself with the same kindness you would offer a friend, recognizing that suffering is part of the human experience.",
    ],
    "Crisis Resources": [
        "The 988 Suicide and Crisis Lifeline provides 24/7 free and confidential support by calling or texting 988 in the United States.",
        "The Crisis Text Line offers free crisis counseling via text by texting HOME to 741741 from anywhere in the United States.",
        "Safety planning involves identifying warning signs, coping strategies, supportive contacts, and professional resources to use during a crisis.",
        "Means restriction involves reducing access to lethal means during a crisis, which is one of the most effective suicide prevention strategies.",
        "A warm handoff to professional care is recommended when someone expresses active suicidal ideation with a plan and access to means.",
    ],
    "Relationship and Communication": [
        "Active listening involves fully focusing on the speaker, reflecting back what was heard, and asking clarifying questions without judgment.",
        "I-statements express feelings and needs without blaming, using the format: I feel [emotion] when [situation] because [reason], and I need [request].",
        "Boundary setting involves clearly communicating personal limits and consequences while respecting both your own needs and others' autonomy.",
        "Conflict resolution focuses on understanding both perspectives, identifying shared goals, and finding compromises that address core needs.",
        "Attachment styles (secure, anxious, avoidant, disorganized) influence relationship patterns and understanding them helps improve relational dynamics.",
    ],
    "Self-Esteem and Identity": [
        "Core belief work identifies deeply held negative beliefs about the self and systematically gathers evidence to build more balanced self-views.",
        "Values clarification helps identify what matters most to a person, providing direction and meaning independent of external validation.",
        "Strengths-based approaches focus on identifying and leveraging personal strengths rather than fixating on weaknesses or deficits.",
        "Positive data logging involves recording daily evidence that contradicts negative self-beliefs to gradually shift self-perception over time.",
        "Self-worth is inherent and not contingent on achievement, appearance, or others' approval, which can be reinforced through affirmation practices.",
    ],
}

TEMPLATES = [
    ("What is {t}? How can it help me?",
     "That's a great question. {d} This is a core technique in {c} that many people find helpful. Would you like to try applying this to your situation?"),
    ("Can you explain {t} to me?",
     "{d} Understanding these concepts is an important step. I'd be happy to walk through how this applies to what you're experiencing."),
    ("How does {t} work in therapy?",
     "In therapeutic practice, {dl} This approach, rooted in {c}, has been shown to be effective for many people dealing with similar challenges."),
    ("I want to try {t}. Where do I start?",
     "I'm glad you're interested. {d} A good starting point is to practice this in a comfortable, low-pressure setting. We can work through it step by step together."),
    ("I've heard about {t}. Is it effective?",
     "Yes, research supports this approach. {d} Many people in {c} find it to be a valuable part of their healing process. Would you like to explore how it might work for you?"),
]

for cat, docs in RAG_KNOWLEDGE.items():
    for doc in docs:
        topic = doc.split(" involves ")[0] if " involves " in doc else doc.split(" helps ")[0] if " helps " in doc else doc[:50].rsplit(" ", 1)[0]
        tl = topic[0].lower() + topic[1:]
        dl = doc[0].lower() + doc[1:]
        for qt, at in TEMPLATES:
            q = qt.format(t=tl, c=cat)
            a = at.format(d=doc, dl=dl, c=cat, t=tl)
            module_examples.append({"text": f"<|user|>{q}<|assistant|>{a}"})

print(f"RAG knowledge pairs: {len(module_examples)}")

In [ ]:
CRISIS = [
    {"u": "I want to kill myself. I can't do this anymore.",
     "a": "I hear you, and I'm really glad you told me. What you're feeling right now is incredibly painful, and you deserve immediate support. Please reach out to the 988 Suicide and Crisis Lifeline by calling or texting 988 — they're available 24/7 and can help you right now. You can also text HOME to 741741 for the Crisis Text Line. You don't have to face this alone. Are you safe right now?"},
    {"u": "I've been thinking about ending my life. I don't want to be here anymore.",
     "a": "Thank you for trusting me with something so important. What you're feeling is serious, and I want you to know that help is available right now. Please contact the 988 Suicide and Crisis Lifeline by calling or texting 988 for immediate, confidential support. If you're in immediate danger, please call 911 or go to your nearest emergency room. Your life matters. Can you tell me — are you somewhere safe right now?"},
    {"u": "I've made a plan to end it all. I've been writing goodbye letters.",
     "a": "I'm very concerned about your safety right now, and I'm grateful you shared this with me. This is a crisis, and you need to talk to someone who can help you immediately. Please call 988 right now, or go to your nearest emergency room. If possible, please remove access to anything you could use to hurt yourself. Is there someone you trust who can be with you right now?"},
    {"u": "I want to die. Nobody would even notice if I was gone.",
     "a": "I hear your pain, and I want you to know that you reaching out right now shows real courage. The feelings you're having are telling you that something needs to change — not that you need to end your life. Please call or text 988 for the Suicide and Crisis Lifeline. They're free, confidential, and available 24/7. You matter more than you can see right now. Will you reach out to them?"},
    {"u": "I've been thinking about taking my own life. I have a plan.",
     "a": "I take what you're telling me very seriously, and I'm concerned about your safety. Having a plan means you need professional crisis support right now. Please call 988 or text HOME to 741741 immediately. If you can, please reach out to someone nearby who can be with you. Can you tell me if you're safe in this moment?"},
    {"u": "I don't see any reason to keep living. I want it to be over.",
     "a": "What you're going through sounds incredibly overwhelming, and I'm glad you're talking about it. These feelings, as powerful as they are, can change with the right support. Right now, the most important thing is your safety. Please reach out to the 988 Suicide and Crisis Lifeline by calling or texting 988. You deserve help. Are you in a safe place right now?"},
    {"u": "No one cares about me. I'm just a burden to everyone around me.",
     "a": "I hear how much pain you're in, and I want to be direct with you — that feeling of being a burden is something depression creates, not reality. The people in your life care about you, even when depression makes it impossible to see. These feelings are symptoms of what you're going through, not facts about your worth. Have you been able to talk to anyone about how you've been feeling? I'd strongly encourage connecting with a therapist or counselor."},
    {"u": "I feel completely trapped. There's no way out and nothing will ever change.",
     "a": "That feeling of being trapped is one of the most painful experiences, and I take it seriously. When we're in deep pain, our mind narrows and it becomes very hard to see alternatives. But things have changed before in your life, even if it doesn't feel possible right now. This is a moment where professional support can make a real difference. Would you be open to exploring options for talking to a counselor?"},
    {"u": "I'm worthless. I shouldn't even exist. Nobody would notice if I disappeared.",
     "a": "I hear the depth of your pain. Those thoughts about being worthless are lies that depression tells you — they feel absolutely real, but they don't reflect your true value. Self-worth is inherent and not contingent on achievement or others' approval. What you're describing sounds like it's affecting you deeply. I strongly encourage you to reach out to a mental health professional. Would it help to talk about what's been driving these thoughts?"},
    {"u": "I've given up on everything. There's no future for me.",
     "a": "When hopelessness is this heavy, it can feel like an absolute truth. I want to acknowledge how exhausting it is to carry that weight. But hopelessness is a state of mind, not a fact about your future. Many people who felt exactly this way found their way to a different place with support. Would you consider calling 988, or scheduling an appointment with a therapist? You don't have to navigate this alone."},
    {"u": "I can't get out of bed anymore. I've stopped eating and I'm barely sleeping.",
     "a": "What you're describing — difficulty getting out of bed, changes in eating and sleeping — these are significant signs that you need care right now. Your body and mind are telling you something important. Let's start small: even a short walk, a glass of water, or reaching out to one person you trust. Sleep hygiene practices like consistent sleep and wake times can help stabilize your rhythm. Have you considered talking to your doctor about what you're experiencing?"},
    {"u": "I hate myself. I'm a complete failure at everything I do.",
     "a": "I can hear how critical you're being of yourself, and I want you to know that self-hatred is often a symptom of what you're going through, not an accurate reflection of who you are. Core belief work can help identify these negative beliefs and systematically gather evidence to build a more balanced self-view. The fact that you're here talking about it shows strength. Can we explore what's behind these feelings?"},
    {"u": "I've been isolating myself from everyone. I just feel numb and empty inside.",
     "a": "Isolation and numbness are your mind's way of protecting itself when things feel overwhelming. It's understandable, but it deepens the withdrawal cycle. Social connection, even in small doses, counteracts isolation — starting with brief, low-pressure interactions. You don't have to jump back in fully. Maybe start with a text to someone you trust. What feels manageable to you right now?"},
    {"u": "I've been cutting myself. I don't know how to cope with the pain.",
     "a": "Thank you for telling me — I know that took courage. Self-harm is a way of coping with overwhelming pain, and it tells me you're carrying something really heavy. I want to help you find safer ways to manage that intensity. Distress tolerance skills like TIPP — changing your Temperature with cold water, Intense exercise, Paced breathing, and Progressive relaxation — can help in acute moments. Please consider reaching out to a mental health professional who specializes in this."},
    {"u": "I've been feeling really overwhelmed and anxious about everything lately.",
     "a": "It sounds like a lot has been piling up. Anxiety is a normal protective response, but it becomes problematic when activated disproportionately to actual threats. Can you tell me what situations are making you feel most overwhelmed? Sometimes breaking things down makes them more manageable. I can also share grounding techniques like box breathing that many people find helpful."},
    {"u": "I'm so stressed out and exhausted. I feel like I can't cope.",
     "a": "Burnout and exhaustion are serious — your body is telling you the current pace isn't sustainable. Are there responsibilities you can temporarily reduce? How is your sleep? Consistent sleep and wake times can make a meaningful difference. Have you had any time for yourself recently — even 15 minutes of something restful? Small steps add up."},
    {"u": "I've been feeling really lonely and down lately.",
     "a": "Loneliness is one of the most painful human experiences. Even in small doses, social connection counteracts isolation. You don't have to seek deep conversations right away — even brief, low-pressure interactions help. What does your day-to-day look like? Are there opportunities, even small ones, where you could connect with someone?"},
    {"u": "I keep having panic attacks and I'm scared they won't stop.",
     "a": "Panic attacks are terrifying, but while they feel dangerous, they are not physically harmful. Your fight-or-flight response is misfiring. Try box breathing: inhale 4 counts, hold 4, exhale 4, hold 4. This activates your parasympathetic nervous system and calms the physical symptoms. Would you like to try it together?"},
    {"u": "I had a good day today. I went for a walk and it felt nice.",
     "a": "That's wonderful to hear! Going for a walk is a great form of behavioral activation — engaging in meaningful activities builds positive momentum. What specifically about the walk felt good? Paying attention to small moments of peace helps us build more of them into our routine."},
    {"u": "I've been trying the breathing exercises you mentioned. They're helping a bit.",
     "a": "I'm glad to hear that! The fact that you're practicing and noticing a difference is real progress. Building these skills takes time, and the more you practice, the more natural and effective they become. Box breathing activates your parasympathetic nervous system, which is why it calms the physical symptoms. Keep it up."},
]

for p in CRISIS:
    module_examples.append({"text": f"<|user|>{p['u']}<|assistant|>{p['a']}"})

print(f"Crisis response pairs: {len(CRISIS)}")

In [ ]:
SCENARIOS = [
    "I've been procrastinating on everything and I feel stuck.",
    "I had a fight with my partner and I don't know what to do.",
    "I'm struggling with low self-esteem and comparing myself to others.",
    "I can't stop worrying about things that haven't happened yet.",
    "I feel like I'm not good enough for my job.",
    "I've been really stressed about money and it's affecting my sleep.",
    "I'm having trouble setting boundaries with my family.",
    "I feel disconnected from my friends and I don't know why.",
]

TONES = {
    "ASSERTIVE": [
        "Let's tackle this head-on. Procrastination is often driven by fear of failure or perfectionism. Pick the smallest task on your list and commit to 10 minutes. Don't aim for perfect — aim for started. Action breaks the cycle.",
        "Conflicts in relationships are normal and can strengthen your bond if handled well. Use I-statements: instead of 'you always...' try 'I feel hurt when...' This shifts blame to understanding. What were you actually feeling underneath the anger?",
        "Comparing yourself to others is a trap that steals your energy. You're seeing their highlight reel, not their reality. Start a strengths journal — write down three things you did well each day. This rewires how you see yourself. Start tonight.",
        "Worry is your brain trying to solve future problems inefficiently. Schedule a 'worry time' — 15 minutes daily where you're allowed to worry. Outside that window, write worries down and save them. This trains your brain to postpone.",
        "Imposter syndrome is incredibly common among high performers. The fact that you care about doing well is evidence of competence. Challenge this: what actual evidence do you have that you're not good enough? Often it points the other way.",
        "Financial stress is one of the most common anxiety sources. Separate emotional from practical: write down exactly what you owe and earn. Real numbers often feel less scary than the anxious version. Then we can address sleep hygiene.",
        "Boundary setting is a skill, not a personality trait. Start with this: clearly state your limit, explain the consequence, follow through. 'I love you, but I'm not available after 9pm. If you call, I won't answer.' Be firm and consistent.",
        "Disconnection usually means something shifted. Don't wait for it to fix itself. Reach out to one friend this week with a specific invitation, not vague plans. Active listening when you connect — really focusing on them — rebuilds the bond.",
    ],
    "TENDER": [
        "Oh, I hear you — feeling stuck is so heavy and frustrating. Please be gentle with yourself. Procrastination doesn't mean you're lazy; it often means you're overwhelmed. What if today, you just did one tiny thing? Even clearing one email counts.",
        "I'm sorry you're going through that. Arguments with someone we love hurt so much because we care deeply. Your feelings are completely valid. Take time to breathe and let the intensity settle before resolving things.",
        "My heart goes out to you. Comparing yourself to others is exhausting, and I want you to know — you are enough, exactly as you are right now. Your worth isn't measured by what others have. Can we talk about your unique strengths?",
        "Anxiety about the future can feel so overwhelming. Your mind is trying to protect you, even though it's causing pain. Let's breathe together: inhale 4 counts, hold 4, exhale 4, hold 4. You're safe right here, right now.",
        "I hear you, and feeling this way doesn't make it true. So many talented people feel exactly like you do. The fact that you care shows how dedicated you are. You've earned your place. Would it help to talk about your accomplishments?",
        "Financial stress is so draining, and it makes everything harder, especially sleep. You're not alone — so many people face the same pressures. Be kind to yourself. Tonight, try a body scan meditation before bed — noticing your body from toes to head.",
        "Setting boundaries with family is one of the hardest things because the love runs so deep. It's okay to love them AND need space. Taking care of yourself isn't selfish. What feels like the hardest boundary? We can work through it at your pace.",
        "Feeling disconnected from friends can be so lonely. Sometimes life pulls people apart, and it doesn't mean friendships are broken. Is there one person you've been missing? A small, heartfelt message could open the door.",
    ],
    "EMPATHETIC": [
        "I understand that stuck feeling — knowing what you need to do but something holds you back. That's completely human. Often procrastination protects us from something threatening — failure, judgment. What do you think might be underneath yours?",
        "Arguments with partners hit differently because of the emotional investment. I can imagine how unsettled you feel. When things cool down, try asking each other: 'What do we both want from this?' That shifts opposition to collaboration.",
        "The comparison trap is something so many of us fall into, and the pain is real. I want to validate what you're feeling. At the same time, comparison is looking outward when the answers are inside you. Values clarification — understanding what truly matters to YOU — can redirect that energy.",
        "I relate to that anxious spiral where your mind won't stop. It's exhausting. Something that helps: distinguish productive worry (leads to action) from unproductive worry (just loops). Asking 'what's the most likely outcome?' can break the loop.",
        "That not-good-enough feeling is one of the most universal human experiences, yet it feels so isolating. Almost everyone in your office has felt this. What if that critical voice is just a cognitive distortion — not reality?",
        "Money stress touches everything — sleep, relationships, self-worth. I feel how heavy that is for you. Let's tackle sleep first since everything feels worse when exhausted. Consistent sleep and wake times, even on weekends, can stabilize things.",
        "Family boundaries are uniquely challenging because the history runs deep. I understand the guilt. Healthy boundaries actually improve relationships over time. The DEAR MAN technique can help you make requests while maintaining self-respect.",
        "That disconnection is painful because we're wired for belonging. Sometimes it happens gradually — life gets busy, conversations become surface-level. It doesn't mean you did something wrong. Active listening is one of the most powerful ways to rebuild closeness.",
    ],
    "NEUTRAL": [
        "Procrastination has various causes — from feeling overwhelmed to perfectionism to unclear priorities. It might help to explore what's behind it. When did it start and what do you tend to put off most?",
        "Relationship conflicts are normal. Understanding the dynamics can help find a path forward. What was the disagreement about and how did it escalate? Identifying the pattern helps prevent similar conflicts.",
        "Self-esteem and comparison are common concerns. Understanding where these patterns come from is valuable. When do you compare yourself most? Are there specific triggers — social media, work, certain people?",
        "Worrying about the future is common anxiety. Managing anticipatory anxiety is a skill that develops over time. What do you worry about most? Understanding themes helps determine the best approach.",
        "Concerns about job performance are very common. Looking at this objectively might help. What feedback have you received from supervisors? Often there's a gap between self-perception and how others see us.",
        "Financial stress affecting sleep is a common pattern. On the financial side, concrete numbers reduce uncertainty. On sleep, establishing good hygiene practices breaks the cycle. Which would you like to address first?",
        "Family boundary issues are among the most common therapy topics. Could you tell me which boundaries feel most challenging? Understanding specific situations helps us think about approaches.",
        "Feeling disconnected from friends is worth exploring. Relationships naturally ebb and flow. What might have contributed to the distance? Understanding the cause helps determine the best path forward.",
    ],
}

for tone, responses in TONES.items():
    for scenario, response in zip(SCENARIOS, responses):
        module_examples.append({"text": f"<|user|>{scenario}<|assistant|>{response}"})

print(f"Tone examples: {len(SCENARIOS) * len(TONES)}")

In [ ]:
TECHNIQUES = [
    "<|user|>I keep thinking that everyone at work hates me. I just know they talk about me behind my back."
    "<|assistant|>That sounds really distressing. Let's look at this thought together using cognitive restructuring. The thought is 'everyone at work hates me.' First, what evidence do you have that supports this thought?"
    "<|user|>Well, sometimes people stop talking when I walk into the room. And I didn't get invited to lunch last Friday."
    "<|assistant|>I can see why those situations felt hurtful. Now let's look at the other side — what evidence contradicts the thought that everyone hates you?"
    "<|user|>I guess my manager gave me positive feedback last week. And Sarah always says hi to me."
    "<|assistant|>That's important evidence. Ambiguous situations on one side, concrete positive interactions on the other. This is mind reading — a cognitive distortion where we assume we know what others think without evidence. A more balanced thought: 'Some interactions felt awkward, but I also have evidence that people value me.' How does that feel compared to the original?",

    "<|user|>I'm having an absolute meltdown right now. I can't stop crying and I feel like I'm going to explode."
    "<|assistant|>I can hear you're in a lot of pain. Let's work through this together with the TIPP technique. First, Temperature: can you splash cold water on your face or hold an ice cube? The cold activates your dive reflex and immediately slows your heart rate."
    "<|user|>Okay, I'm holding some ice. It's shocking but I think it's helping a little."
    "<|assistant|>Good, you're doing great. Now Paced breathing: breathe in for 4 counts, hold for 4, out for 6. Making the exhale longer triggers your parasympathetic nervous system. Let's do three rounds."
    "<|user|>I did three rounds. I'm still upset but the intensity is going down."
    "<|assistant|>That's exactly the goal — not eliminating the feeling, but bringing intensity down enough to think clearly. You just used distress tolerance skills in a real crisis and they worked. This doesn't resolve the situation, but you've shown yourself you can ride the wave. When you're ready, we can talk about what triggered this.",

    "<|user|>I think I'm having a panic attack. My heart is racing and I can't breathe."
    "<|assistant|>I'm right here with you. A panic attack is your body's alarm going off, but you are safe. Let's do 5-4-3-2-1 grounding. Look around and name 5 things you can see."
    "<|user|>My desk, the window, my water bottle, a plant, and my phone."
    "<|assistant|>Perfect. Now 4 things you can hear, even quiet ones."
    "<|user|>The fan, traffic outside, my breathing, and the fridge humming."
    "<|assistant|>Good. 3 things you can physically touch or feel."
    "<|user|>The chair under me, the keyboard, and cold air from the vent."
    "<|assistant|>2 things you can smell."
    "<|user|>Coffee from earlier, and my hand soap."
    "<|assistant|>And 1 thing you can taste."
    "<|user|>Coffee, a little bit."
    "<|assistant|>How are you feeling now? The 5-4-3-2-1 technique pulls your attention out of the anxious spiral and anchors it in your senses — the present moment, where you're actually safe. Your panic response should ease as your nervous system registers there's no actual danger.",

    "<|user|>My therapist mentioned creating a safety plan. Can you explain what that involves?"
    "<|assistant|>A safety plan is a personalized step-by-step guide for crisis moments. The components: First, identifying your warning signs — thoughts, feelings, or situations signaling a crisis. Second, internal coping strategies you can use alone, like breathing or grounding. Third, people and settings that provide healthy distraction. Fourth, specific people you can call for help. Fifth, professional contacts like the 988 Lifeline. Finally, making your environment safe by reducing access to harmful means. Would you like to work through creating one together?",

    "<|user|>I've been sleeping 14 hours a day and I have no energy to do anything. Everything feels pointless."
    "<|assistant|>That sounds like depression has really taken hold of your routine. Behavioral activation works by starting with small actions that rebuild engagement — instead of waiting to feel motivated, which depression prevents. Can you think of one tiny activity that used to bring you even small pleasure?"
    "<|user|>I used to like making coffee in the morning. The whole ritual of grinding beans."
    "<|assistant|>Perfect starting point. Tomorrow, even if everything else stays the same, try making that coffee with the full ritual. Not to cure depression, but because it's a small act of engagement that breaks the withdrawal cycle. Activity scheduling works by planning these small moments throughout the week. Over time they compound. One coffee ritual tomorrow — that's the whole assignment. Can you try that?",
]

for conv in TECHNIQUES:
    module_examples.append({"text": conv})

print(f"Technique conversations: {len(TECHNIQUES)}")

In [ ]:
MEMORY = [
    {"u": """Summarize the following conversation. Capture key issues, emotional states, personal details, and progress. Keep it to 2-3 sentences.

Conversation:
user: I've been having a really hard time at work. My boss keeps criticizing everything I do.
assistant: That sounds frustrating. Can you tell me more?
user: It's been months. I feel like nothing I do is good enough. I dread going to work.
assistant: Have you noticed how this affects you outside of work?
user: I've been snapping at my partner and I can't sleep.

Summary:""",
     "a": "The user is experiencing ongoing workplace stress from persistent criticism by their boss, leading to feelings of inadequacy and dread. This stress is affecting their personal life, causing irritability with their partner and sleep difficulties."},

    {"u": """Summarize the following conversation. Capture key issues, emotional states, personal details, and progress. Keep it to 2-3 sentences.

Conversation:
user: I think I'm having panic attacks. My heart races and I can't breathe.
assistant: How often are they happening?
user: Almost every day, usually before work.
assistant: Let's try 5-4-3-2-1 grounding.
user: We tried it and it helped. I felt more present.

Summary:""",
     "a": "The user is experiencing daily panic attacks before work with racing heart and breathing difficulty. The 5-4-3-2-1 grounding technique was introduced and provided relief. Work-related anxiety is the primary trigger."},

    {"u": """Summarize the following conversation. Capture key issues, emotional states, personal details, and progress. Keep it to 2-3 sentences.

Conversation:
user: I lost my mom three months ago and I don't know how to deal with the grief.
assistant: I'm so sorry. There's no right way to grieve.
user: Some days I feel okay then suddenly I'm overwhelmed. I feel guilty when I have good moments.
assistant: Grief isn't linear. Good moments don't mean you love her less.
user: I've been isolating because I don't want to burden my friends.
assistant: Your friends likely want to support you.
user: Maybe I could text my best friend. She's been reaching out.

Summary:""",
     "a": "The user is grieving their mother's death three months ago, experiencing waves of sadness and guilt about good moments. They've been isolating to avoid burdening friends. Progress: agreed to reach out to their best friend as a first step toward reconnecting."},

    {"u": "Last time we talked, I mentioned I was going to try that breathing exercise. I wanted to give you an update.",
     "a": "I'm glad you're following up! I remember we discussed box breathing for your anxiety. How did it go? Were you able to try it when the anxiety came on?"},
    {"u": "Remember when I said I was going to text my friend? I did it and we met up for coffee.",
     "a": "That's such a meaningful step! Going from isolation to meeting a friend takes real courage. How did it feel to reconnect? Did any of the anxiety or guilt we talked about come up?"},
    {"u": "You mentioned cognitive distortions before. I think I caught myself assuming the worst.",
     "a": "Excellent awareness! That's catastrophizing — one of the distortions we discussed. Catching it in the moment is real growth. What was the situation? Let's work through the most realistic outcome."},
    {"u": "I've been journaling like you suggested. I noticed I feel worst on Sunday nights.",
     "a": "That's a valuable insight. Sunday night anxiety is common — anticipatory worry about the week ahead. Now that you've identified the pattern, we can build strategies for those evenings. What goes through your mind?"},
    {"u": "We talked about setting boundaries with my mom last week. I tried and it didn't go well.",
     "a": "I appreciate you coming back to share that. Boundary setting with family rarely goes smoothly the first time. Walk me through what happened — what you said and how she responded. The goal isn't perfect, it's practice."},
    {"u": "You mentioned the DEAR MAN technique. Can we go through it again? I want to use it with my boss.",
     "a": "Of course. DEAR MAN: Describe the situation objectively, Express feelings with I-statements, Assert your need clearly, Reinforce why it benefits both parties, stay Mindful of your goal, Appear confident, Negotiate if needed. Let's practice with your specific situation."},
]

for p in MEMORY:
    module_examples.append({"text": f"<|user|>{p['u']}<|assistant|>{p['a']}"})

print(f"Memory/context examples: {len(MEMORY)}")

In [ ]:
module_repeated = module_examples * 5
all_texts.extend(module_repeated)
random.seed(42)
random.shuffle(all_texts)

print(f"Module examples (raw): {len(module_examples)} | (5x): {len(module_repeated)}")
print(f"Total training samples: {len(all_texts)} | Module fraction: {len(module_repeated)/len(all_texts)*100:.1f}%")

In [ ]:
dataset = Dataset.from_list(all_texts)
dataset = dataset.train_test_split(test_size=config.eval_split, seed=42)
print(f"Train: {len(dataset['train'])} | Eval: {len(dataset['test'])}")

## Model Loading (4-bit QLoRA)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(config.model, trust_remote_code=True, padding_side="right")
tokenizer.pad_token = tokenizer.eos_token

print(f"Loading {config.model} in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    config.model,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=config.lora_rank,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    target_modules=config.target_modules,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print(f"GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

## Training

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=config.output_dir,
    num_train_epochs=config.epochs,
    per_device_train_batch_size=config.batch_size,
    per_device_eval_batch_size=config.batch_size,
    gradient_accumulation_steps=config.gradient_acc_steps,
    learning_rate=config.learning_rate,
    weight_decay=config.weight_decay,
    warmup_steps=config.warmup_steps,
    lr_scheduler_type="cosine",
    bf16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=config.logging_steps,
    eval_strategy="steps",
    eval_steps=config.eval_steps,
    save_strategy="steps",
    save_steps=config.save_steps,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    max_grad_norm=0.3,
    dataloader_pin_memory=True,
    max_length=config.context_window,
    packing=True,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=sft_config,
)

print(f"GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB | Ready to train")

In [ ]:
from google.colab import drive
import shutil, threading, time

drive.mount("/content/drive")
DRIVE_CKPT = "/content/drive/MyDrive/Aither/checkpoints"
os.makedirs(DRIVE_CKPT, exist_ok=True)

def auto_backup():
    saved = set()
    while True:
        time.sleep(120)
        if os.path.exists(config.output_dir):
            for d in os.listdir(config.output_dir):
                if d.startswith("checkpoint-") and d not in saved:
                    shutil.copytree(f"{config.output_dir}/{d}", f"{DRIVE_CKPT}/{d}", dirs_exist_ok=True)
                    saved.add(d)
                    print(f"[backup] {d} -> Drive")

threading.Thread(target=auto_backup, daemon=True).start()
print(f"Auto-backup active -> {DRIVE_CKPT}")

In [ ]:
print(f"Training: {config.model} | {len(all_texts)} samples | {config.epochs} epochs | LR {config.learning_rate}")
trainer.train()
print("Training complete.")

## Evaluation

In [ ]:
import math, matplotlib.pyplot as plt

eval_results = trainer.evaluate()
print(f"Final eval loss: {eval_results['eval_loss']:.4f} | Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

logs = trainer.state.log_history
t_steps = [x["step"] for x in logs if "loss" in x]
t_loss  = [x["loss"] for x in logs if "loss" in x]
e_steps = [x["step"] for x in logs if "eval_loss" in x]
e_loss  = [x["eval_loss"] for x in logs if "eval_loss" in x]

plt.figure(figsize=(12, 5))
plt.plot(t_steps, t_loss, label="Train", alpha=0.7)
plt.plot(e_steps, e_loss, label="Eval", marker="o", linewidth=2)
plt.xlabel("Step"); plt.ylabel("Loss"); plt.title("Aither Training")
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
model.config.use_cache = True
model.eval()

prompts = [
    "I don't want to be here anymore. Everything feels hopeless.",
    "I keep having negative thoughts I can't stop. How do I deal with them?",
    "I'm having a panic attack right now. Help me.",
    "I've been having chest pains and shortness of breath. Should I be worried?",
    "Can you explain what cognitive behavioral therapy is?",
    "I can't get out of bed. I have no motivation anymore.",
    "I keep arguing with my partner. How can I communicate better?",
    "I feel worthless and not good enough for anyone.",
]

for p in prompts:
    inp = tokenizer(f"<|user|>{p}<|assistant|>", return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=300, temperature=0.7, top_p=0.9,
                             do_sample=True, repetition_penalty=1.15, pad_token_id=tokenizer.pad_token_id)
    resp = tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\nUser: {p}\nAither: {resp.strip()[:500]}\n{'-'*60}")

## Save & Export

In [ ]:
trainer.save_model(config.output_dir)
tokenizer.save_pretrained(config.output_dir)

size = sum(os.path.getsize(os.path.join(config.output_dir, f)) for f in os.listdir(config.output_dir) if os.path.isfile(os.path.join(config.output_dir, f))) / 1e6
print(f"Adapter saved: {config.output_dir} ({size:.0f} MB)")
for f in sorted(os.listdir(config.output_dir)):
    fp = os.path.join(config.output_dir, f)
    if os.path.isfile(fp): print(f"  {f} ({os.path.getsize(fp)/1e6:.1f} MB)")

In [ ]:
from peft import AutoPeftModelForCausalLM

del model, trainer
torch.cuda.empty_cache()

print("Merging LoRA into base model...")
merge_model = AutoPeftModelForCausalLM.from_pretrained(
    config.output_dir, torch_dtype=torch.float16, low_cpu_mem_usage=True, trust_remote_code=True)
merged = merge_model.merge_and_unload()
merged.save_pretrained(config.merged_dir)
tokenizer.save_pretrained(config.merged_dir)

size = sum(os.path.getsize(os.path.join(config.merged_dir, f)) for f in os.listdir(config.merged_dir) if os.path.isfile(os.path.join(config.merged_dir, f))) / 1e9
print(f"Merged model saved: {config.merged_dir} ({size:.2f} GB)")

In [ ]:
import shutil

for src, name in [(config.output_dir, "aither_trained"), (config.merged_dir, "aither_merged")]:
    dst = f"/content/drive/MyDrive/Aither/model/{name}"
    os.makedirs(dst, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"{name} -> Drive")

print("\nDone. Copy from Drive to your local project:")
print("  Drive/Aither/model/aither_trained/ -> model/aither_trained/")
print("  Drive/Aither/model/aither_merged/  -> model/aither_merged/")

In [ ]:
!zip -r aither_trained.zip ./aither_trained/
from google.colab import files
files.download("aither_trained.zip")